# Shape toroidal AutoEncoder
latent space $\mathbb{R}^D / {A\mathbb{Z}^D$ with $A$ learnable

In [469]:
import os
import sys

mvae_dir = os.path.split(os.getcwd())[0]
if mvae_dir not in sys.path:
    sys.path.append(mvae_dir)

In [470]:
%pwd
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Imports

In [471]:
import torch
import numpy as np
import torch.optim as optim

import lib.dataloaders as dataloaders
import lib.models as models
from lib.trainer import AETrainer
import lib.utils as utils



### Set up and initialize data loader

In [472]:
# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

### Dataloader

In [473]:
batch_size = 64

_, pointcloud, pointcloud_embedded = dataloaders.generate_toroidal_pointcloud(A=A, embed_dim=embed_dim, num_points=20000, translation=translation)
dataloader = dataloaders.load_shaped_torus(pointcloud_embedded,batch_size)

pointcloud not translated. Translation invalid tensor([0]). Shape torch.Size([1]), Instance True


### Model

In [474]:
data_dim = embed_dim
hidden_dim1 = 8
hidden_dim2 = 4
latent_dim = A.shape[0]
activation = torch.nn.ReLU

shape_model = models.ShapeToroidalAE(data_dim=data_dim, hidden_dim1=hidden_dim1, hidden_dim2=hidden_dim2, latent_dim=latent_dim, activation=activation)
euclidean_model = models.EuclideanAE(data_dim=data_dim, hidden_dim1=hidden_dim1, hidden_dim2=hidden_dim2, latent_dim=2*latent_dim)

### Optimizer

In [475]:
learning_rate = 0.001

shape_optimizer = optim.Adam(shape_model.parameters(), lr=learning_rate)
euclidean_optimizer = optim.Adam(euclidean_model.parameters(), lr=learning_rate)

### Trainer setup

In [476]:
num_epochs = 15
log_interval = 100
device = "cpu"
dataset = "synthetic"
show_latents = False

trainer_config = {'num_epochs': num_epochs, 'log_interval': log_interval, 'device': device, 'dataset': dataset, 'show_latents':show_latents }

train_loader, test_loader = dataloader


### Train and evaluate shape model

In [ ]:
ae_history = AETrainer(shape_model, dataloader, shape_optimizer, trainer_config).train()

Trainer successfully initialized.
Start training...
Epoch 0
Step [100/282], Loss: 1.6782, Shape matrix A:tensor([[ 1.8553, -0.7974],
        [ 0.3940, -2.3733]], grad_fn=<LinalgInvExBackward0>), A_inv_T:tensor([[ 0.5804, -0.1950],
        [ 0.0963, -0.4537]], grad_fn=<MmBackward0>)


In [ ]:
utils.plot_latent_projections(shape_model, pointcloud, train_loader)

### Train and evaluate euclidean model

In [ ]:
euclid_ae_history = AETrainer(euclidean_model, dataloader, euclidean_optimizer, trainer_config).train()

In [ ]:
utils.plot_latent_projections(euclidean_model, pointcloud, train_loader)